# Task 1-ChatBot 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Input,Embedding,LSTM,Dense,GlobalMaxPooling1D,Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

In [ ]:
data1={"intents": [
        {"tag": "greeting",
         "patterns": ["Hi there", "How are you", "Is anyone there?","Hey","Hola", "Hello", "Good day"],
         "responses": ["Hello", "Good to see you again", "Hi there, how can I help?"],
         "context": [""]
        },
        {"tag": "goodbye",
         "patterns": ["Bye", "See you later", "Goodbye", "Nice chatting to you, bye", "Till next time"],
         "responses": ["See you!", "Have a nice day", "Bye! Come back again soon."],
         "context": [""]
        },
        {"tag": "ML",
         "patterns": ["What is machine learning"],
         "responses": ["Machine learning is a branch of artificial intelligence (AI) and computer science which focuses on the use of data and algorithms to imitate the way that humans learn, gradually improving its accuracy"],
         "context": [""]
        },
        {"tag": "powerful",
         "patterns": ["powerful country in the world?"],
         "responses": ["The United States of America is a North American nation that is the world's most dominant economic and military power."],
         "context": [""]
        },
        {"tag": "thanks",
         "patterns": ["Thanks", "Thank you", "That's helpful", "Awesome, thanks", "Thanks for helping me"],
         "responses": ["My pleasure", "You're Welcome"],
         "context": [""]
        },
        {"tag": "query",
         "patterns": ["What is big bang?"],
         "responses": ["The Big Bang theory is the prevailing cosmological model explaining the existence of the observable universe from the earliest known periods through its subsequent large-scale evolution."],
         "context": [""]
        } 
    ]
 }

In [ ]:
tags=[]
patterns=[]
responses={}
for intent in data1['intents']:
    responses[intent['tag']]=intent['responses']
    for lines in intent['patterns']:
        patterns.append(lines)
        tags.append(intent['tag'])
    

In [ ]:
data=pd.DataFrame({'patterns':patterns,'tags':tags})

In [ ]:
data

In [ ]:
data=data.sample(frac=1)

In [ ]:
import string
data['patterns']=data['patterns'].apply(lambda wrd:[ltrs.lower() for ltrs in wrd if ltrs not in string.punctuation])


In [ ]:
data['patterns']=data['patterns'].apply(lambda wrd:''.join(wrd))

In [ ]:
data

In [ ]:
tokenzier=Tokenizer(num_words=2000)
tokenzier.fit_on_texts(data['patterns'])
train=tokenzier.texts_to_sequences(data['patterns'])

In [ ]:
#Apply padding
x_train=pad_sequences(train)

In [ ]:
le=LabelEncoder()
y_train=le.fit_transform(data['tags'])

In [ ]:
input_shape=x_train.shape[1]
input_shape

In [ ]:
vocabulary=len(tokenzier.word_index)

In [ ]:
output_length=le.classes_.shape[0]

In [ ]:
i=Input(shape=(input_shape,))
x=Embedding(vocabulary+1,10)(i)
x=LSTM(10,return_sequences=True)(x)
x=Flatten()(x)
x=Dense(output_length,activation='softmax')(x)
model=Model(i,x)

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",optimizer='adam',metrics=['accuracy'])

In [ ]:
train=model.fit(x_train,y_train,epochs=200)

In [ ]:
plt.plot(train.history['accuracy'],label='training set accuracy')
plt.plot(train.history['loss'],label='training set loss')
plt.legend()

In [ ]:
import random
while True:
    texts_p=[]
    prediction_input=input('You : ')
 
    prediction_input=[letters.lower() for letters in prediction_input if letters not in string.punctuation]
    prediction_input=''.join(prediction_input)
    texts_p.append(prediction_input)

    prediction_input=tokenzier.texts_to_sequences(texts_p)
    prediction_input=np.array(prediction_input).reshape(-1)
    prediction_input=pad_sequences([prediction_input],input_shape)

    output=model.predict(prediction_input)
    output=output.argmax()
 
    response_tag=le.inverse_transform([output])[0]
    print("chatbot : ",random.choice(responses[response_tag]))
    if response_tag=='goodbye':
        break